# Poster campaign · 2/3 — Computation and statistics

**Stage A:** one expensive calculation of dense \(\mathcal O_1\) SI signals and the full `legend_poster` background, with `factorized_shm` and `vectorized` folding.

**Stage B:** generate **27 sensitivity configurations** (3 backgrounds × 3 thresholds × 3 exposures) from the saved per-kg-day binned spectra using the unchanged `PoissonCLsCounting` implementation.

Do not launch until the speedup branch is merged and independently validated. Interruptions can be restarted with the same cached physics inputs; changes in code may intentionally invalidate the cache.

In [6]:
from pathlib import Path
import sys

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'analyses/legend_poster').is_dir():
            return candidate
    raise RuntimeError('Open this notebook in the WIMP-LEGEND checkout (or a subdirectory).')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# CAMPAIGN_DIR = ROOT / 'analyses/legend_poster/campaigns/poster_final'
CAMPAIGN_DIR = ROOT / 'analyses/legend_poster/campaigns/poster_sigma20'
SPEC_PATH = ROOT / 'analyses/legend_poster/poster_campaign_spec.json'
CONFIG_PATH = CAMPAIGN_DIR / 'config.json'
print('Repository:', ROOT)
print('Campaign:', CAMPAIGN_DIR)


Repository: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code
Campaign: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code\analyses\legend_poster\campaigns\poster_sigma20


## Validate saved inputs and optimization switches

In [7]:
import json
import numpy as np
from analyses.legend_poster import load_config, resolve_profile
from analyses.legend_poster.campaign_scripts.poster_campaign import load_spec

if not CONFIG_PATH.exists():
    raise FileNotFoundError('Run poster_setup_campaign.ipynb first: ' + str(CONFIG_PATH))
spec = load_spec(SPEC_PATH)
config = load_config(CONFIG_PATH)
resolved = resolve_profile(config, 'standard')
if resolved['signal'].get('si_rate_engine') != 'factorized_shm':
    raise RuntimeError('Fast SI backend is not enabled — refuse a potentially multi-hour generic run.')
if resolved['numerical'].get('folding_backend') != 'vectorized':
    raise RuntimeError('Vectorized folding backend is not enabled — refuse slow fallback.')
if resolved['signal']['operators'] != [1] or resolved['analysis']['scenarios'] != ['legend_poster']:
    raise RuntimeError('Unexpected source physics run dimensions.')
print('Masses:', len(resolved['signal']['masses_GeV']))
print('Optimization:', resolved['signal']['si_rate_engine'], resolved['numerical']['folding_backend'])
print('Background source:', spec['input_scenario'])


Masses: 79
Optimization: factorized_shm vectorized
Background source: legend_poster


## Stage A — Compute/reuse source spectra

Computes full physical spectra and **one preliminary 100 kg·yr limit per mass** (the latter is incidental). Mass-dependent spectra and each background component are cached at unit exposure. Keep the output/run directory distinct from the official validation snapshots.

`RECOMPUTE_SOURCE=False` can reuse a previously recorded source run by verifying the source index. It does not skip verification of its saved checksums.

In [8]:
from analyses.legend_poster import run_analysis, load_run
import time

SOURCE_INDEX = CAMPAIGN_DIR / 'source_run.json'
RUNS_DIR = CAMPAIGN_DIR / 'runs'
CACHE_DIR = ROOT / 'analyses/legend_poster/cache'
RECOMPUTE_SOURCE = True  # False = use exact saved source_run.json if present

if RECOMPUTE_SOURCE:
    start = time.perf_counter()
    run = run_analysis(config, profile='standard', cache_dir=CACHE_DIR, output_dir=RUNS_DIR)
    elapsed_s = time.perf_counter() - start
    SOURCE_INDEX.parent.mkdir(parents=True,exist_ok=True)
    SOURCE_INDEX.write_text(json.dumps({'run_relative': str(run['path'].resolve().relative_to(ROOT)),
                                       'run_id': run['manifest']['run_id']},indent=2)+'\n')
    print(f'Stage A elapsed: {elapsed_s:.1f} s; cache hits/misses:',
          run['manifest']['cache']['hits'], run['manifest']['cache']['misses'])
else:
    if not SOURCE_INDEX.exists():
        raise FileNotFoundError('No source_run.json. Set RECOMPUTE_SOURCE=True.')
    index=json.loads(SOURCE_INDEX.read_text())
    run = load_run(ROOT/index['run_relative'])
    if run['manifest']['run_id'] != index['run_id']:
        raise ValueError('Source index run ID mismatch')
print('Verified numerical output:',run['path'])
print('Available saved masses:',len(run['manifest']['resolved_configuration']['signal']['masses_GeV']))


Stage A elapsed: 424.9 s; cache hits/misses: 316 325
Verified numerical output: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code\analyses\legend_poster\campaigns\poster_sigma20\runs\ed8ac71f232a961dc2f3
Available saved masses: 79


## Stage B — All 27 sensitivity configurations

All thresholds are **observed electron-equivalent analysis cuts**. `none` means no modeled background, `neutrinos` sums the existing `solar_CEvNS` and `solar_ES` components, and `full` uses the complete saved benchmark. The exact existing 90%-CL median counting-statistics backend is called for each mass/combination. No signal folding or background integration happens in this stage.

In [9]:
from analyses.legend_poster.campaign_scripts.poster_campaign import build_limits
start = time.perf_counter()
rows, poster_manifest = build_limits(run['path'], SPEC_PATH, CAMPAIGN_DIR)
print(f'Stage B completed in {time.perf_counter() - start:.2f} s')
print('Masses:', poster_manifest['n_masses'], '| limits:', poster_manifest['n_limits'])
print('Exposures [t·yr]:',poster_manifest['exposure_tonne_year'])
print('Thresholds [keVee]:',poster_manifest['threshold_keVee'])
print('Backgrounds:',poster_manifest['background_modes'])
print('Saved:', CAMPAIGN_DIR/'poster_limits.csv')


Stage B completed in 2.39 s
Masses: 79 | limits: 2133
Exposures [t·yr]: [0.1, 1.0, 10.0]
Thresholds [keVee]: [0.1, 0.15, 1.0]
Backgrounds: ['none', 'neutrinos', 'full']
Saved: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code\analyses\legend_poster\campaigns\poster_sigma20\poster_limits.csv


## Quick inspection of expected limits and numerical convergence

In [10]:
from analyses.legend_poster.campaign_scripts.poster_campaign import select_curve
for bg in ('none','neutrinos','full'):
    x,y=select_curve(rows,bg,0.1,10.0)
    subset=[(float(x[i]),float(y[i])) for i in range(len(x))
            if np.isfinite(y[i]) and 2 <= x[i] <= 10]
    print(bg, 'first 5 finite 2–10 GeV points:',subset[:5])
checks=run['convergence']
assert checks and all(ch['passed'] for ch in checks), 'Source-fold convergence failure'
print('Numerical convergence checks passed:',len(checks))
print('Numerical run ID:',run['manifest']['run_id'])


none first 5 finite 2–10 GeV points: [(2.0, 1.781555494378325e-45), (2.0321827859810995, 1.4641126757441677e-45), (2.165929206240564, 6.8781293210646296e-46), (2.30847803593663, 3.6144274949285216e-46), (2.460408598326993, 2.0024891998107e-46)]
neutrinos first 5 finite 2–10 GeV points: [(2.0, 3.024099291773153e-44), (2.0321827859810995, 2.485256350287901e-44), (2.165929206240564, 1.2783457398157868e-44), (2.30847803593663, 6.717652103260746e-45), (2.460408598326993, 4.0350848207352805e-45)]
full first 5 finite 2–10 GeV points: [(2.0, 7.682686995275949e-44), (2.0321827859810995, 6.313763140722988e-44), (2.165929206240564, 3.36667719444141e-44), (2.30847803593663, 1.7691744441138736e-44), (2.460408598326993, 1.0859445705879318e-44)]
Numerical convergence checks passed: 167
Numerical run ID: ed8ac71f232a961dc2f342429f59a2eddad5bbffa708f1eb31f81aa4bccaa970
